In [1]:
import pymysql


conn = pymysql.connect(
    host="localhost",
    user="root",
    password=""   
)


In [2]:
cursor = conn.cursor()


cursor.execute("CREATE DATABASE IF NOT EXISTS hospital_db")
cursor.execute("USE hospital_db")

0

In [3]:
cursor.execute("""
CREATE TABLE IF NOT EXISTS patients (
    patient_id INT AUTO_INCREMENT PRIMARY KEY,
    name VARCHAR(100),
    age INT,
    gender VARCHAR(10),
    phone VARCHAR(15)
)
""")

cursor.execute("""
CREATE TABLE IF NOT EXISTS specializations (
    specialization_id INT AUTO_INCREMENT PRIMARY KEY,
    specialization_name VARCHAR(50)
)
""")

cursor.execute("""
CREATE TABLE IF NOT EXISTS doctors (
    doctor_id INT AUTO_INCREMENT PRIMARY KEY,
    name VARCHAR(100),
    specialization_id INT,
    FOREIGN KEY (specialization_id) REFERENCES specializations(specialization_id)
)
""")

cursor.execute("""
CREATE TABLE IF NOT EXISTS appointments (
    appointment_id INT AUTO_INCREMENT PRIMARY KEY,
    patient_id INT,
    doctor_id INT,
    appointment_date DATE,
    status VARCHAR(20),
    FOREIGN KEY (patient_id) REFERENCES patients(patient_id),
    FOREIGN KEY (doctor_id) REFERENCES doctors(doctor_id)
)
""")

cursor.execute("""
CREATE TABLE IF NOT EXISTS billing (
    bill_id INT AUTO_INCREMENT PRIMARY KEY,
    appointment_id INT,
    amount DECIMAL(10,2),
    status VARCHAR(20),
    FOREIGN KEY (appointment_id) REFERENCES appointments(appointment_id)
)
""")

cursor.execute("""
CREATE TABLE IF NOT EXISTS audit_log (
    log_id INT AUTO_INCREMENT PRIMARY KEY,
    action VARCHAR(255),
    action_time TIMESTAMP DEFAULT CURRENT_TIMESTAMP
)
""")
cursor.execute("""
    CREATE TABLE IF NOT EXISTS doctor_visits (
    visit_id INT AUTO_INCREMENT PRIMARY KEY,
    patient_id INT,
    doctor_id INT,
    visit_date TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
    diagnosis VARCHAR(255),
    prescription VARCHAR(255),

    FOREIGN KEY (patient_id) REFERENCES patients(patient_id),
    FOREIGN KEY (doctor_id) REFERENCES doctors(doctor_id)
);
""")

conn.commit()

In [4]:
cursor.execute("DROP PROCEDURE IF EXISTS book_appointment")
cursor.execute("""
CREATE PROCEDURE book_appointment(
    IN p_id INT,
    IN d_id INT,
    IN a_date DATE
)
BEGIN
    DECLARE doctor_count INT;
    DECLARE appointment_count INT;

    -- Check if doctor exists
    SELECT COUNT(*) INTO doctor_count
    FROM doctors
    WHERE doctor_id = d_id;

    IF doctor_count = 0 THEN
        SIGNAL SQLSTATE '45000'
        SET MESSAGE_TEXT = 'Doctor does not exist';

    ELSE
        -- Count appointments for that doctor on that date
        SELECT COUNT(*) INTO appointment_count
        FROM appointments
        WHERE doctor_id = d_id
        AND appointment_date = a_date;

        -- Check limit
        IF appointment_count >= 10 THEN
            SIGNAL SQLSTATE '45000'
            SET MESSAGE_TEXT = 'Doctor is fully booked for this date. Try another date';

        ELSE
            -- Insert appointment
            INSERT INTO appointments(patient_id, doctor_id, appointment_date, status)
            VALUES(p_id, d_id, a_date, 'Scheduled');

        END IF;

    END IF;

END;
""")

cursor.execute("DROP PROCEDURE IF EXISTS generate_bill")
cursor.execute("""
CREATE PROCEDURE generate_bill(IN app_id INT)
BEGIN
    INSERT INTO billing(appointment_id, amount, status)
    VALUES(app_id, 500.00, 'Pending');
END
""")


conn.commit()

In [5]:
cursor.execute("DROP TRIGGER IF EXISTS after_appointment_insert")
cursor.execute("""
CREATE TRIGGER after_appointment_insert
AFTER INSERT ON appointments
FOR EACH ROW
BEGIN
    CALL generate_bill(NEW.appointment_id);
END
""")

cursor.execute("DROP TRIGGER IF EXISTS log_patient_insert")
cursor.execute("""
CREATE TRIGGER log_patient_insert
AFTER INSERT ON patients
FOR EACH ROW
BEGIN
    INSERT INTO audit_log(action)
    VALUES(CONCAT('New patient added: ', NEW.name));
END
""")

conn.commit()

In [6]:
cursor.execute("""
CREATE TABLE IF NOT EXISTS roles (
    role_id INT AUTO_INCREMENT PRIMARY KEY,
    role_name VARCHAR(50) UNIQUE NOT NULL
);
""")
cursor.execute("""
INSERT IGNORE INTO roles VALUES
(1,'admin'),(2,'doctor'),(3,'patient');
""")
cursor.execute("""
CREATE TABLE IF NOT EXISTS users (
    user_id INT AUTO_INCREMENT PRIMARY KEY,
    username VARCHAR(50) UNIQUE NOT NULL,
    password_hash BLOB NOT NULL,
    role_id INT,
    reference_id INT,  -- links to patient_id or doctor_id
    created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP,

    FOREIGN KEY (role_id) REFERENCES roles(role_id)
);
""")

0

In [7]:

import bcrypt

conn = pymysql.connect(host="localhost", user="root", password="", database="hospital_db")
cursor = conn.cursor()

def create_user(username, password, role_id, ref_id=None):
    hashed = bcrypt.hashpw(password.encode(), bcrypt.gensalt())

    cursor.execute("""
        INSERT INTO users (username, password_hash, role_id, reference_id)
        VALUES (%s, %s, %s, %s)
    """, (username, hashed, role_id, ref_id))

    conn.commit()


create_user("admin1", "1234", 1, None)


In [8]:
cursor.execute("""
INSERT IGNORE INTO specializations ( specialization_name)
VALUES 
('General Physician'),
('Cardiology'),
('Neurology'),
('Orthopedics'),
('Dermatology'),
('Pediatrics'),
('Psychiatry'),
('Pulmonology'),
('Gastroenterology'),
('Nephrology'),
('General Surgery'),
('Neurosurgery'),
('Cardiothoracic Surgery'),
('Plastic Surgery'),
('Vascular Surgery'),
('Gynecology'),
('Obstetrics'),
('Ophthalmology'),
('ENT'),
('Radiology'),
('Pathology'),
('Anesthesiology'),
('Emergency Medicine'),
('Oncology'),
('Endocrinology'),
('Rheumatology'),
('Urology'),
('Hematology'),
('Infectious Diseases'),
('Geriatrics'),
('Physiotherapy'),
('Nutrition'),
('Clinical Psychology');
""")
conn.commit()

In [9]:
#Demo data for DB

import pymysql
import bcrypt
import random

conn = pymysql.connect(
    host="localhost",
    user="root",
    password="",
    database="hospital_db"
)
cursor = conn.cursor()

# ================= SPECIALIZATIONS =================
specializations = [
'General Physician','Cardiology','Neurology','Orthopedics','Dermatology',
'Pediatrics','Psychiatry','Pulmonology','Gastroenterology','Nephrology',
'General Surgery','Neurosurgery','Cardiothoracic Surgery','Plastic Surgery',
'Vascular Surgery','Gynecology','Obstetrics','Ophthalmology','ENT','Radiology',
'Pathology','Anesthesiology','Emergency Medicine','Oncology','Endocrinology',
'Rheumatology','Urology','Hematology','Infectious Diseases','Geriatrics',
'Physiotherapy','Nutrition','Clinical Psychology'
]

# Insert specializations safely
for spec in specializations:
    cursor.execute(
        "INSERT IGNORE INTO specializations (specialization_name) VALUES (%s)",
        (spec,)
    )

# Fetch specialization ids
cursor.execute("SELECT specialization_id, specialization_name FROM specializations")
spec_map = {name: sid for sid, name in cursor.fetchall()}

# ================= PASSWORD HASH =================
def hash_pw(p):
    return bcrypt.hashpw(p.encode(), bcrypt.gensalt())

# ================= DOCTORS =================
doctor_ids = []

for i, spec in enumerate(specializations, start=1):
    doc_name = f"Dr. {spec.split()[0]}_{i}"

    cursor.execute(
        "INSERT INTO doctors (name, specialization_id) VALUES (%s,%s)",
        (doc_name, spec_map[spec])
    )
    did = cursor.lastrowid
    doctor_ids.append(did)

    # Create user login for doctor
    cursor.execute("""
        INSERT INTO users (username,password_hash,role_id,reference_id)
        VALUES (%s,%s,2,%s)
    """, (f"doctor{did}", hash_pw("1234"), did))

# ================= PATIENTS =================
names = ["Rahul","Amit","Priya","Neha","Ravi","Anjali","Karan","Sneha","Arjun","Pooja"]

patient_ids = []

for i in range(20):
    name = random.choice(names) + f"_{i}"
    age = random.randint(18, 60)
    gender = random.choice(["Male","Female"])
    phone = "9" + str(random.randint(100000000, 999999999))

    cursor.execute("""
        INSERT INTO patients (name, age, gender, phone)
        VALUES (%s,%s,%s,%s)
    """, (name, age, gender, phone))

    pid = cursor.lastrowid
    patient_ids.append(pid)

    # Create login for patient
    cursor.execute("""
        INSERT INTO users (username,password_hash,role_id,reference_id)
        VALUES (%s,%s,3,%s)
    """, (f"patient{pid}", hash_pw("1234"), pid))

# ================= ADMIN (IF NOT EXISTS) =================
cursor.execute("SELECT * FROM users WHERE username='admin'")
if not cursor.fetchone():
    cursor.execute("""
        INSERT INTO users (username,password_hash,role_id,reference_id)
        VALUES (%s,%s,1,NULL)
    """, ("admin", hash_pw("admin123")))

conn.commit()

print("✅ Doctors, Patients, Users inserted successfully!")


✅ Doctors, Patients, Users inserted successfully!


In [6]:

import tkinter as tk
from tkinter import ttk, messagebox
from tkinter import filedialog
import joblib
import pymysql
import bcrypt

# ================= DB =================
conn = pymysql.connect(
    host="localhost",
    user="root",
    password="",
    database="hospital_db"
)
cursor = conn.cursor(pymysql.cursors.DictCursor)

current_user = None

# ================= COMMON =================
def clear_screen():
    for widget in root.winfo_children():
        widget.destroy()

def add_back_button(parent, role):
    tk.Button(parent, text="⬅ Back", command=lambda: open_dashboard(role)).pack(pady=10)

def show_table(data, columns, role):
    clear_screen()
    frame = tk.Frame(root)
    frame.pack(fill="both", expand=True)

    tree = ttk.Treeview(frame, columns=columns, show="headings")

    for col in columns:
        tree.heading(col, text=col)
        tree.column(col, width=120)

    for row in data:
        tree.insert("", "end", values=[row[col] for col in columns])

    tree.pack(fill="both", expand=True)
    add_back_button(frame, role)

# ================= AUTH =================
def hash_password(password):
    return bcrypt.hashpw(password.encode(), bcrypt.gensalt())

def check_password(password, hashed):
    if isinstance(hashed, str):
        hashed = hashed.encode()
    return bcrypt.checkpw(password.encode(), hashed)

# ================= LOGIN =================
def login():
    global current_user

    username = username_entry.get()
    password = password_entry.get()
    role = role_var.get()

    cursor.execute("SELECT * FROM users WHERE username=%s", (username,))
    user = cursor.fetchone()

    if user and check_password(password, user['password_hash']):
        cursor.execute("SELECT role_name FROM roles WHERE role_id=%s", (user['role_id'],))
        db_role = cursor.fetchone()['role_name']

        if db_role != role:
            messagebox.showerror("Error", "Wrong role selected")
            return

        current_user = user
        open_dashboard(role)
    else:
        messagebox.showerror("Error", "Invalid credentials")

# ================= SIGNUP =================
def signup():
    signup_win = tk.Toplevel(root)
    signup_win.title("New Patient Registration")
    signup_win.geometry("400x400")

    # ===== INPUT FIELDS =====
    tk.Label(signup_win, text="Full Name").pack()
    name_entry = tk.Entry(signup_win)
    name_entry.pack()

    tk.Label(signup_win, text="Age").pack()
    age_entry = tk.Entry(signup_win)
    age_entry.pack()

    tk.Label(signup_win, text="Gender").pack()
    gender_var = tk.StringVar()
    ttk.Combobox(signup_win, textvariable=gender_var,
                 values=["Male", "Female", "Other"]).pack()

    tk.Label(signup_win, text="Phone").pack()
    phone_entry = tk.Entry(signup_win)
    phone_entry.pack()

    tk.Label(signup_win, text="Username").pack()
    username_entry = tk.Entry(signup_win)
    username_entry.pack()

    tk.Label(signup_win, text="Password").pack()
    password_entry = tk.Entry(signup_win, show="*")
    password_entry.pack()

    # ===== SUBMIT FUNCTION =====
    def register():
        name = name_entry.get().strip()
        age = age_entry.get().strip()
        gender = gender_var.get().strip()
        phone = phone_entry.get().strip()
        username = username_entry.get().strip()
        password = password_entry.get().strip()

        # ===== VALIDATION =====
        if not (name and age and gender and phone and username and password):
            messagebox.showerror("Error", "All fields are required")
            return

        if not age.isdigit():
            messagebox.showerror("Error", "Age must be a number")
            return

        # check duplicate username
        cursor.execute("SELECT * FROM users WHERE username=%s", (username,))
        if cursor.fetchone():
            messagebox.showerror("Error", "Username already exists")
            return

        try:
            hashed = hash_password(password)

            # insert patient
            cursor.execute("""
                INSERT INTO patients (name, age, gender, phone)
                VALUES (%s, %s, %s, %s)
            """, (name, age, gender, phone))

            patient_id = cursor.lastrowid

            # insert user
            cursor.execute("""
                INSERT INTO users (username, password_hash, role_id, reference_id)
                VALUES (%s, %s, 3, %s)
            """, (username, hashed, patient_id))

            conn.commit()

            messagebox.showinfo("Success", "Patient Registered Successfully")
            signup_win.destroy()

        except Exception as e:
            messagebox.showerror("Error", str(e))

    tk.Button(signup_win, text="Register", command=register).pack(pady=10)

# ================= DASHBOARD =================
def open_dashboard(role):
    clear_screen()

    frame = tk.Frame(root)
    frame.pack(fill="both", expand=True)

    tk.Label(frame, text=f"{role.capitalize()} Dashboard", font=("Arial", 16)).pack(pady=10)

    if role == "patient":
        tk.Button(frame, text="Predict Disease", command=predict_disease_ui).pack()
        tk.Button(frame, text="Check Pneumonia (Upload X-ray)", command=pneumonia_ui).pack()
        tk.Button(frame, text="View Doctors", command=view_doctors).pack()
        tk.Button(frame, text="Book Appointment", command=book_appointment).pack()
        tk.Button(frame, text="View Bills", command=view_bills).pack()

    elif role == "doctor":
        tk.Button(frame, text="My Appointments", command=view_my_appointments).pack()

    elif role == "admin":
        tk.Button(frame, text="View Doctors", command=view_doctors).pack()
        tk.Button(frame, text="Add Doctor", command=add_doctor).pack()
        tk.Button(frame, text="View Logs", command=view_logs).pack()
        tk.Button(frame, text="Doctor Visits", command=view_visits).pack()

# ================= PATIENT =================
def view_doctors():
    cursor.execute("""
        SELECT d.doctor_id, d.name, s.specialization_name
        FROM doctors d
        JOIN specializations s ON d.specialization_id=s.specialization_id
    """)
    data = cursor.fetchall()
    show_table(data, ["doctor_id","name","specialization_name"], current_role())

def book_appointment():
    clear_screen()
    frame = tk.Frame(root)
    frame.pack()

    tk.Label(frame, text="Doctor ID").pack()
    did_entry = tk.Entry(frame)
    did_entry.pack()

    tk.Label(frame, text="Date (YYYY-MM-DD)").pack()
    date_entry = tk.Entry(frame)
    date_entry.pack()

    def submit():
        try:
            cursor.callproc('book_appointment', [
                current_user['reference_id'],
                did_entry.get(),
                date_entry.get()
            ])
            conn.commit()
            messagebox.showinfo("Success", "Booked")
        except Exception as e:
            messagebox.showerror("Error", str(e))

    tk.Button(frame, text="Book", command=submit).pack()
    add_back_button(frame, "patient")

def view_bills():
    pid = current_user['reference_id']

    cursor.execute("""
        SELECT * FROM billing
        WHERE appointment_id IN (
            SELECT appointment_id FROM appointments WHERE patient_id=%s
        )
    """, (pid,))

    data = cursor.fetchall()
    show_table(data, ["bill_id","appointment_id","amount","status"], "patient")






# ================= PNEUMONIA MODEL =================
import os
os.environ["TORCHINDUCTOR_CACHE_DIR"] = "C:/temp/torch_cache"
import torch
import torch.nn as nn
import torchvision.transforms as transforms
from torchvision import models
from PIL import Image
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

pneumonia_model = models.resnet50(weights=None)
pneumonia_model.fc = nn.Linear(pneumonia_model.fc.in_features, 2)

pneumonia_model.load_state_dict(
    torch.load(r"C:\HospitalProject\models\chest_xray_model.pth", map_location=device)
)

pneumonia_model = pneumonia_model.to(device)
pneumonia_model.eval()

pneumonia_classes = ["NORMAL", "PNEUMONIA"]

# Transform (same as training!)
pneumonia_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])



def pneumonia_ui():
    clear_screen()
    frame = tk.Frame(root)
    frame.pack()

    tk.Label(frame, text="Upload Chest X-ray Image", font=("Arial", 14)).pack(pady=10)

    file_path_var = tk.StringVar()

    def browse_file():
        path = filedialog.askopenfilename(
            filetypes=[("Image Files", "*.jpg *.jpeg *.png")]
        )
        file_path_var.set(path)

    tk.Button(frame, text="Select Image", command=browse_file).pack(pady=5)

    tk.Label(frame, textvariable=file_path_var, wraplength=400).pack()

    def predict():
        path = file_path_var.get()

        if not path:
            messagebox.showerror("Error", "Please select an image")
            return

        try:
            image = Image.open(path).convert("RGB")
            image = pneumonia_transform(image).unsqueeze(0).to(device)

            with torch.no_grad():
                outputs = pneumonia_model(image)
                probs = torch.softmax(outputs, dim=1)
                conf, pred = torch.max(probs, 1)

            result = pneumonia_classes[pred.item()]
            confidence = conf.item() * 100

            messagebox.showinfo(
                "Prediction Result",
                f"Result: {result}\nConfidence: {confidence:.2f}%"
            )

        except Exception as e:
            messagebox.showerror("Error", str(e))

    tk.Button(frame, text="Predict", command=predict).pack(pady=10)

    add_back_button(frame, "patient")





def predict_disease_ui():
    clear_screen()
    frame = tk.Frame(root)
    frame.pack(fill="both", expand=True)

    tk.Label(frame, text="Select Symptoms", font=("Arial", 14)).pack(pady=10)

    # Load dataset to get feature names
    import pandas as pd
    df = pd.read_csv(r"C:\Users\PINTU SHAW\Downloads\improved_disease_dataset.csv")
    features = df.columns[:-1]

    symptom_vars = {}

    # Scrollable frame (important if many symptoms)
    canvas = tk.Canvas(frame)
    scrollbar = tk.Scrollbar(frame, orient="vertical", command=canvas.yview)
    scroll_frame = tk.Frame(canvas)

    scroll_frame.bind(
        "<Configure>",
        lambda e: canvas.configure(scrollregion=canvas.bbox("all"))
    )

    canvas.create_window((0, 0), window=scroll_frame, anchor="nw")
    canvas.configure(yscrollcommand=scrollbar.set)

    canvas.pack(side="left", fill="both", expand=True)
    scrollbar.pack(side="right", fill="y")

    # Create checkboxes
    for f in features:
        var = tk.IntVar()
        chk = tk.Checkbutton(scroll_frame, text=f, variable=var)
        chk.pack(anchor="w")
        symptom_vars[f] = var

    # ===== Prediction Function =====
    import joblib
    import numpy as np

    model = joblib.load(r"C:\HospitalProject\models\disease_model.pkl")
    label_encoder = joblib.load(r"C:\HospitalProject\models\label_encoder.pkl")
    def predict():
        try:
            input_data = [symptom_vars[f].get() for f in features]
            input_array = np.array(input_data).reshape(1, -1)

            # Get probabilities
            probs = model.predict_proba(input_array)[0]

            # Top 3 predictions
            top3_idx = np.argsort(probs)[-3:][::-1]

            result_text = "Top Predictions:\n\n"
            for i in top3_idx:
                disease = label_encoder.inverse_transform([i])[0]
                confidence = probs[i] * 100
                result_text += f"{disease} → {confidence:.2f}%\n"

            messagebox.showinfo("Prediction Result", result_text)

        except Exception as e:
            messagebox.showerror("Error", str(e))

    tk.Button(frame, text="Predict", command=predict).pack(pady=10)
    add_back_button(frame, "patient")

# ================= DOCTOR =================
def view_my_appointments():
    did = current_user['reference_id']

    cursor.execute("SELECT * FROM appointments WHERE doctor_id=%s", (did,))
    data = cursor.fetchall()

    clear_screen()
    frame = tk.Frame(root)
    frame.pack(fill="both", expand=True)

    columns = ["appointment_id","patient_id","appointment_date","status"]
    tree = ttk.Treeview(frame, columns=columns, show="headings")

    for col in columns:
        tree.heading(col, text=col)

    for row in data:
        tree.insert("", "end", values=[row[col] for col in columns])

    tree.pack()

    def complete():
        selected = tree.selection()
        if not selected:
            return

        vals = tree.item(selected[0])['values']
        mark_completed(vals[0], vals[1], did)

    tk.Button(frame, text="Mark Completed", command=complete).pack()
    add_back_button(frame, "doctor")

def mark_completed(app_id, pid, did):
    popup = tk.Toplevel(root)

    tk.Label(popup, text="Diagnosis").pack()
    diag = tk.Entry(popup)
    diag.pack()

    tk.Label(popup, text="Prescription").pack()
    pres = tk.Entry(popup)
    pres.pack()

    def save():
        cursor.execute("UPDATE appointments SET status='Completed' WHERE appointment_id=%s", (app_id,))
        cursor.execute("""
            INSERT INTO doctor_visits(patient_id,doctor_id,diagnosis,prescription)
            VALUES (%s,%s,%s,%s)
        """, (pid, did, diag.get(), pres.get()))

        conn.commit()
        popup.destroy()
        messagebox.showinfo("Done", "Completed")

    tk.Button(popup, text="Save", command=save).pack()

# ================= ADMIN =================
def add_doctor():
    clear_screen()
    frame = tk.Frame(root)
    frame.pack()

    tk.Label(frame, text="Doctor Name").pack()
    name_entry = tk.Entry(frame)
    name_entry.pack()

    cursor.execute("SELECT * FROM specializations")
    specs = cursor.fetchall()

    spec_var = tk.StringVar()
    ttk.Combobox(
        frame,
        textvariable=spec_var,
        values=[f"{s['specialization_id']} - {s['specialization_name']}" for s in specs]
    ).pack()

    def submit():
        try:
            spec_id = spec_var.get().split(" - ")[0]

            cursor.execute(
                "INSERT INTO doctors (name,specialization_id) VALUES (%s,%s)",
                (name_entry.get(), spec_id)
            )
            did = cursor.lastrowid

            hashed = hash_password("1234")

            cursor.execute("""
                INSERT INTO users(username,password_hash,role_id,reference_id)
                VALUES (%s,%s,2,%s)
            """, (f"doctor{did}", hashed, did))

            conn.commit()
            messagebox.showinfo("Success", "Doctor added")

        except Exception as e:
            messagebox.showerror("Error", str(e))

    tk.Button(frame, text="Add", command=submit).pack()
    add_back_button(frame, "admin")

def view_logs():
    cursor.execute("SELECT * FROM audit_log")
    show_table(cursor.fetchall(), ["log_id","action","action_time"], "admin")

def view_visits():
    cursor.execute("SELECT * FROM doctor_visits")
    show_table(cursor.fetchall(),
               ["visit_id","patient_id","doctor_id","visit_date","diagnosis","prescription"],
               "admin")

# ================= HELP =================
def current_role():
    cursor.execute("SELECT role_name FROM roles WHERE role_id=%s", (current_user['role_id'],))
    return cursor.fetchone()['role_name']

# ================= UI =================
root = tk.Tk()
root.title("Hospital System")
root.geometry("700x500")

login_frame = tk.Frame(root)
login_frame.pack()

tk.Label(login_frame, text="Username").pack()
username_entry = tk.Entry(login_frame)
username_entry.pack()

tk.Label(login_frame, text="Password").pack()
password_entry = tk.Entry(login_frame, show="*")
password_entry.pack()

role_var = tk.StringVar()
ttk.Combobox(login_frame, textvariable=role_var,
             values=["patient","doctor","admin"]).pack()

tk.Button(login_frame, text="Login", command=login).pack()
tk.Button(login_frame, text="New Patient? Register Here", command=signup).pack(pady=5)

root.mainloop()



c:\testenv\myenv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
